# CI7340 Applied Data Programming — Unified Data Cleaning & Fact-Verification Pipeline (Group 17)

**Input Dataset:** `data/raw/dataset_coursework1.csv` (`508,106` offence records × `30` columns)  
**External GIS Layers:**
- `data/external/montgomery_county_police_districts.geojson` (`7` official Montgomery County Police District polygons, `EPSG:4326`)
- `data/external/maryland_zip_codes.geojson` (`445` official Maryland ZIP Code `MultiPolygon` geometries, `EPSG:4326`)

**Target Output:** `data/processed/cleaned_crime_data.csv.gz`  
**Reference Documents:**
- `docs/Dataset_Quality_Analysis.md` (raw quality audit)
- `docs/Data_Cleaning_Options_Checklist.md` (14 locked team decisions)
- `docs/Data_Cleaning_Plan.md` (approved pre-execution specification)

### Design Principles of This Notebook
1. **One Logical Operation Per Cell:** Each code cell performs exactly one atomic, verifiable cleaning step.
2. **Evidence-First Fact Verification:** Every step prints **BEFORE** evidence (proving *why* cleaning is required), executes the transformation, and prints **AFTER** evidence (proving the issue is resolved without side effects).
3. **Non-Destructive Auditability:** Raw columns (`City`, `Place`, `Agency`, `State`, `Zip Code`, `Dispatch Date / Time`, `Victims`) are preserved alongside their `_Clean` / `_Group` / provenance flag counterparts so every decision can be audited or sensitivity-tested.
4. **Strict Library Compliance:** Uses only Group 17's approved libraries (`numpy`, `pandas`, `geopandas`) plus Python standard library (`pathlib`, `re`, `difflib`).

## Step 1 — Environment Setup, Imports & File Verification

**Why:** Establish deterministic file paths relative to the repository root and confirm that the raw crime CSV and both official GIS boundary files exist before loading any data.

In [1]:
from pathlib import Path
import re
import difflib
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)

# Resolve repository root cleanly regardless of whether notebook runs from repo root or notebooks/
CWD = Path.cwd().resolve()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

RAW_CSV = REPO_ROOT / "data" / "raw" / "dataset_coursework1.csv"
RAW_GZ = REPO_ROOT / "data" / "raw" / "dataset_coursework1.csv.gz"
RAW_DATA_PATH = RAW_CSV if RAW_CSV.exists() else RAW_GZ

DISTRICTS_GEOJSON = REPO_ROOT / "data" / "external" / "montgomery_county_police_districts.geojson"
ZIPS_GEOJSON = REPO_ROOT / "data" / "external" / "maryland_zip_codes.geojson"
OUTPUT_PATH = REPO_ROOT / "data" / "processed" / "cleaned_crime_data.csv.gz"

for label, p in [
    ("Raw Dataset", RAW_DATA_PATH),
    ("Police Districts GeoJSON", DISTRICTS_GEOJSON),
    ("Maryland ZIP Codes GeoJSON", ZIPS_GEOJSON),
]:
    assert p.exists(), f"Missing required file: {p}"
    print(f"[OK] {label:28s}: {p.relative_to(REPO_ROOT)} ({p.stat().st_size / 1024 / 1024:.2f} MB)")

print(f"pandas={pd.__version__} | numpy={np.__version__} | geopandas={gpd.__version__}")

[OK] Raw Dataset                 : data/raw/dataset_coursework1.csv (176.43 MB)
[OK] Police Districts GeoJSON    : data/external/montgomery_county_police_districts.geojson (2.44 MB)
[OK] Maryland ZIP Codes GeoJSON  : data/external/maryland_zip_codes.geojson (21.55 MB)
pandas=3.1.1 | numpy=2.5.5 | geopandas=1.2.0


## Step 2 — Load Raw Dataset & Capture Baseline Evidence Snapshot

**Why:** Preserve identifier/code columns (`Offence Code`, `CR Number`, `PRA`) as strings on load so leading zeros (e.g., `PRA = '094'`) and alphabetic codes (`Offence Code = 'FTAS'`, `'NSUP'`) are never corrupted. Capture an immutable `BASELINE` metrics dictionary for the final Before-vs-After comparison table.

In [2]:
# Load raw dataset while preserving string identifiers with leading zeros or alpha characters
df = pd.read_csv(
    RAW_DATA_PATH,
    dtype={"Offence Code": str, "CR Number": str, "PRA": str},
    low_memory=False,
)

# Capture baseline statistics for final Before-vs-After audit
BASELINE = {
    "rows": len(df),
    "columns": df.shape[1],
    "exact_duplicates": int(df.duplicated().sum()),
    "unique_incident_ids": int(df["Incident ID"].nunique()),
    "dup_incident_offence_key": int(df.duplicated(subset=["Incident ID", "Offence Code"]).sum()),
    "null_island_coords": int(((df["Latitude"] == 0.0) & (df["Longitude"] == 0.0)).sum()),
    "missing_end_time": int(df["End_Date_Time"].isna().sum()),
    "missing_dispatch_time": int(df["Dispatch Date / Time"].isna().sum()),
    "unknown_district_rows": int((df["Police District Number"] == "Unknown").sum()),
    "corrupted_beat_own": int((df["Beat"] == "own").sum()),
    "corrupted_sector_w": int((df["Sector"] == "w").sum()),
    "missing_or_non_md_state": int((df["State"].isna() | (df["State"] != "MD")).sum()),
    "missing_or_invalid_zip": int((df["Zip Code"].isna() | (df["Zip Code"] < 10000) | (df["Zip Code"] > 99999)).sum()),
    "unique_cities": int(df["City"].nunique(dropna=True)),
    "legacy_place_rows": int((df["Place"] == "School/College - DO NOT USE").sum()),
    "double_space_block_addr": int(df["Block Address"].dropna().str.contains(r"  ", regex=True).sum()),
    "invalid_addr_num": int(((df["Address Number"] == 0) | (df["Address Number"] > 50000)).sum()),
    "cryptic_agency_p": int((df["Agency"] == "P").sum()),
}

print("=== BASELINE RAW DATASET PROFILE ===")
print(f"Shape                        : {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Exact duplicate rows         : {BASELINE['exact_duplicates']:,}")
print(f"Unique Incident IDs          : {BASELINE['unique_incident_ids']:,} ({len(df) - BASELINE['unique_incident_ids']:,} multi-offence rows)")
print(f"Duplicate (Incident, Offence): {BASELINE['dup_incident_offence_key']:,} (confirmed composite primary key)")
print(f"Total NaN cells              : {int(df.isna().sum().sum()):,} / {df.size:,} ({df.isna().sum().sum() / df.size * 100:.2f}%)")

=== BASELINE RAW DATASET PROFILE ===
Shape                        : 508,106 rows x 30 columns
Exact duplicate rows         : 0
Unique Incident IDs          : 467,601 (40,505 multi-offence rows)
Duplicate (Incident, Offence): 0 (confirmed composite primary key)
Total NaN cells              : 1,439,929 / 15,243,180 (9.45%)


## Step 3 — Load & Verify Official External GIS Polygons

**Why:** Before performing any `geopandas` point-in-polygon spatial joins (Items #6, #7, #8), verify that both boundary layers load cleanly in `EPSG:4326`, contain valid polygon geometries, and expose the expected attribute columns (`AREA_P` for Police Districts; `zipcode1` for Maryland ZIP Codes).

In [3]:
gdf_districts = gpd.read_file(DISTRICTS_GEOJSON).to_crs("EPSG:4326")
gdf_zips = gpd.read_file(ZIPS_GEOJSON).to_crs("EPSG:4326")

# Official mapping of Montgomery County AREA_P integer codes to District Number & Name
AREA_P_TO_DISTRICT = {
    1: ("1D", "ROCKVILLE"),
    2: ("2D", "BETHESDA"),
    3: ("3D", "SILVER SPRING"),
    4: ("4D", "WHEATON"),
    5: ("5D", "GERMANTOWN"),
    6: ("6D", "MONTGOMERY VILLAGE"),
    8: ("TPPD", "TAKOMA PARK"),
}

gdf_districts["dist_num_poly"] = gdf_districts["AREA_P"].map(lambda x: AREA_P_TO_DISTRICT[int(x)][0])
gdf_districts["dist_name_poly"] = gdf_districts["AREA_P"].map(lambda x: AREA_P_TO_DISTRICT[int(x)][1])

# Clean 5-digit ZIP column from Maryland ZIP boundary layer
gdf_zips["zip_poly"] = gdf_zips["zipcode1"].astype(str).str.strip().str.zfill(5)

print("=== POLICE DISTRICTS POLYGON LAYER ===")
print(gdf_districts[["AREA_P", "dist_num_poly", "dist_name_poly"]].sort_values("AREA_P").to_string(index=False))
print(f"Districts CRS: {gdf_districts.crs} | Invalid geometries: {(~gdf_districts.is_valid).sum()}")

print("\n=== MARYLAND ZIP CODES POLYGON LAYER ===")
print(f"Total ZIP polygons : {len(gdf_zips):,} | Unique ZIPs: {gdf_zips['zip_poly'].nunique():,}")
print(f"ZIPs CRS           : {gdf_zips.crs} | Invalid geometries: {(~gdf_zips.is_valid).sum()}")

=== POLICE DISTRICTS POLYGON LAYER ===
 AREA_P dist_num_poly     dist_name_poly
      1            1D          ROCKVILLE
      2            2D           BETHESDA
      3            3D      SILVER SPRING
      4            4D            WHEATON
      5            5D         GERMANTOWN
      6            6D MONTGOMERY VILLAGE
      8          TPPD        TAKOMA PARK
Districts CRS: EPSG:4326 | Invalid geometries: 5

=== MARYLAND ZIP CODES POLYGON LAYER ===
Total ZIP polygons : 445 | Unique ZIPs: 445
ZIPs CRS           : EPSG:4326 | Invalid geometries: 4


## Step 4 — [Item #1] Sanitise Null Island Coordinates `(0.0, 0.0)`

**Why:** In the raw dataset, `Latitude` and `Longitude` report `0` explicit `NaN`s, yet **`24,648` rows (`4.85%`)** are encoded as `(0.0, 0.0)` ("Null Island" in the Atlantic Ocean) due to failed geocoding or privacy redaction. Leaving `0.0` in place corrupts map bounding boxes, centroids, and KDE heatmaps.  
**Decision (Option A):** Replace `0.0` with `NaN` and add boolean flag `has_valid_coords`. Retain all `508,106` rows so non-spatial counts remain 100% accurate.

In [4]:
# BEFORE evidence
null_island_mask = (df["Latitude"] == 0.0) & (df["Longitude"] == 0.0)
print(f"BEFORE — Null Island (0.0, 0.0) rows : {int(null_island_mask.sum()):,} ({null_island_mask.mean()*100:.2f}%)")
print(f"BEFORE — Lat range                   : [{df['Latitude'].min():.5f}, {df['Latitude'].max():.5f}]")
print(f"BEFORE — Lon range                   : [{df['Longitude'].min():.5f}, {df['Longitude'].max():.5f}]")

# Apply cleaning: replace 0.0 with NaN and create explicit flag
df.loc[null_island_mask, ["Latitude", "Longitude"]] = np.nan
df["has_valid_coords"] = df["Latitude"].notna() & df["Longitude"].notna()

# AFTER evidence & Montgomery County bounding-box check
valid_lat = df.loc[df["has_valid_coords"], "Latitude"]
valid_lon = df.loc[df["has_valid_coords"], "Longitude"]
out_of_moco = ((valid_lat < 38.90) | (valid_lat > 39.38) | (valid_lon < -77.55) | (valid_lon > -76.85)).sum()

print(f"\nAFTER  — Null Island (0.0, 0.0) rows : {int(((df['Latitude'] == 0) | (df['Longitude'] == 0)).sum()):,}")
print(f"AFTER  — has_valid_coords == True    : {int(df['has_valid_coords'].sum()):,} ({df['has_valid_coords'].mean()*100:.2f}%)")
print(f"AFTER  — Valid Lat range             : [{valid_lat.min():.5f}, {valid_lat.max():.5f}]")
print(f"AFTER  — Valid Lon range             : [{valid_lon.min():.5f}, {valid_lon.max():.5f}]")
print(f"AFTER  — Valid coords outside MoCo   : {int(out_of_moco):,} (100% inside Montgomery County)")

BEFORE — Null Island (0.0, 0.0) rows : 24,648 (4.85%)
BEFORE — Lat range                   : [0.00000, 39.35124]
BEFORE — Lon range                   : [-77.51670, 0.00000]

AFTER  — Null Island (0.0, 0.0) rows : 0
AFTER  — has_valid_coords == True    : 483,458 (95.15%)
AFTER  — Valid Lat range             : [38.93839, 39.35124]
AFTER  — Valid Lon range             : [-77.51670, -76.90530]
AFTER  — Valid coords outside MoCo   : 0 (100% inside Montgomery County)


## Step 5 — Parse Temporal Columns (`Start_Date_Time`, `End_Date_Time`, `Dispatch Date / Time`)

**Why:** All three timestamp columns are stored as 12-hour strings (`MM/DD/YYYY HH:MM:SS AM/PM`). Converting them with an explicit `format="%m/%d/%Y %I:%M:%S %p"` guarantees fast, unambiguous parsing and proves zero malformed date strings exist.

In [5]:
DATE_COLS = ["Start_Date_Time", "End_Date_Time", "Dispatch Date / Time"]
DATE_FMT = "%m/%d/%Y %I:%M:%S %p"

print("=== BEFORE PARSING ===")
for col in DATE_COLS:
    print(f"  {col:22s} | dtype={str(df[col].dtype):8s} | nulls={int(df[col].isna().sum()):,}")

print("\n=== AFTER PARSING ===")
for col in DATE_COLS:
    orig_nulls = int(df[col].isna().sum())
    df[col] = pd.to_datetime(df[col], format=DATE_FMT, errors="coerce")
    new_nulls = int(df[col].isna().sum())
    parse_failures = new_nulls - orig_nulls
    assert parse_failures == 0, f"Unexpected parse failures in {col}: {parse_failures}"
    print(f"  {col:22s} | dtype={str(df[col].dtype):14s} | parse_fail={parse_failures} | range=[{df[col].min()} -> {df[col].max()}]")

=== BEFORE PARSING ===
  Start_Date_Time        | dtype=str      | nulls=0
  End_Date_Time          | dtype=str      | nulls=282,444
  Dispatch Date / Time   | dtype=str      | nulls=73,405

=== AFTER PARSING ===


  Start_Date_Time        | dtype=datetime64[us] | parse_fail=0 | range=[2016-07-01 00:00:00 -> 2026-09-14 20:44:00]


  End_Date_Time          | dtype=datetime64[us] | parse_fail=0 | range=[2016-07-01 00:00:00 -> 2026-09-14 19:00:00]


  Dispatch Date / Time   | dtype=datetime64[us] | parse_fail=0 | range=[2017-04-02 07:37:54 -> 2026-09-14 19:46:02]


## Step 6 — [Item #3] Check & Guard Same-Incident `End_Date_Time` Backfill

**Why:** `End_Date_Time` is missing on `282,444` rows (`55.59%`). Before accepting those nulls as structural (point-in-time incidents), we verify whether any multi-offence incident (`Incident ID`) has an `End_Date_Time` populated on one offence row and missing on another.  
**Decision (Option D):** Backfill missing `End_Date_Time` only when another row with the same `Incident ID` has a single unambiguous `End_Date_Time`; otherwise keep `NaT` as-is (Option A).

In [6]:
# BEFORE evidence: check if any null End_Date_Time can be recovered from the same Incident ID
inc_end_nunique = df.groupby("Incident ID")["End_Date_Time"].transform("nunique")
recoverable_end_mask = df["End_Date_Time"].isna() & (inc_end_nunique == 1)
n_recoverable_end = int(recoverable_end_mask.sum())

print(f"BEFORE — Missing End_Date_Time rows                   : {int(df['End_Date_Time'].isna().sum()):,}")
print(f"CHECK  — Recoverable from same Incident ID            : {n_recoverable_end:,}")

if n_recoverable_end > 0:
    inc_end_val = df.groupby("Incident ID")["End_Date_Time"].transform("first")
    df.loc[recoverable_end_mask, "End_Date_Time"] = inc_end_val[recoverable_end_mask]

print(f"AFTER  — Missing End_Date_Time rows (structural NaT)  : {int(df['End_Date_Time'].isna().sum()):,}")
print("FACT   — All offences sharing an Incident ID already share identical End_Date_Time status; 0 rows needed backfill.")

BEFORE — Missing End_Date_Time rows                   : 282,444
CHECK  — Recoverable from same Incident ID            : 0
AFTER  — Missing End_Date_Time rows (structural NaT)  : 282,444
FACT   — All offences sharing an Incident ID already share identical End_Date_Time status; 0 rows needed backfill.


## Step 7 — [Item #2] Resolve Negative Crime Durations (`Start_Date_Time > End_Date_Time`)

**Why:** `8,418` rows (`3.73%` of rows with an `End_Date_Time`) have `End_Date_Time < Start_Date_Time`, producing impossible negative durations. Our diagnostic evidence proves **`77.44%` (`6,519` rows)** occur because the officer entered an end *date* without a time, causing the RMS to default to `12:00:00 AM` (`00:00:00`) on the same calendar day.  
**Decision (Option A):** Flag `has_invalid_end_time`, nullify inverted `End_Date_Time` → `NaT` (leaving `Start_Date_Time` untouched), and compute `duration_mins` only on valid non-negative windows.

In [7]:
# BEFORE evidence
neg_dur_mask = df["End_Date_Time"].notna() & (df["End_Date_Time"] < df["Start_Date_Time"])
same_day_midnight = (
    neg_dur_mask
    & (df["Start_Date_Time"].dt.date == df["End_Date_Time"].dt.date)
    & (df["End_Date_Time"].dt.hour == 0)
    & (df["End_Date_Time"].dt.minute == 0)
)
BASELINE["negative_durations"] = int(neg_dur_mask.sum())

print(f"BEFORE — Inverted rows (Start > End)                  : {int(neg_dur_mask.sum()):,}")
print(f"         of which same-day 00:00:00 midnight defaults : {int(same_day_midnight.sum()):,} ({same_day_midnight.sum()/neg_dur_mask.sum()*100:.2f}%)")
print(f"         of which other chronological inversions      : {int((neg_dur_mask & ~same_day_midnight).sum()):,}")

# Apply Option A
df["has_invalid_end_time"] = neg_dur_mask
df.loc[neg_dur_mask, "End_Date_Time"] = pd.NaT
df["duration_mins"] = (df["End_Date_Time"] - df["Start_Date_Time"]).dt.total_seconds() / 60.0

# AFTER evidence
after_neg = int((df["duration_mins"] < 0).sum())
valid_dur = df["duration_mins"].dropna()
print(f"\nAFTER  — Negative durations remaining                 : {after_neg:,}")
print(f"AFTER  — Valid duration_mins count                    : {len(valid_dur):,}")
print(f"AFTER  — Valid duration_mins summary (mins)           : min={valid_dur.min():.1f}, median={valid_dur.median():.1f}, p90={valid_dur.quantile(0.90):.1f}, max={valid_dur.max():.1f}")

BEFORE — Inverted rows (Start > End)                  : 8,418
         of which same-day 00:00:00 midnight defaults : 6,519 (77.44%)
         of which other chronological inversions      : 1,899



AFTER  — Negative durations remaining                 : 0
AFTER  — Valid duration_mins count                    : 217,244
AFTER  — Valid duration_mins summary (mins)           : min=0.0, median=300.0, p90=8460.0, max=5032800.0


## Step 8 — [Item #4] Clean `Dispatch Date / Time` & Build Sensitivity-Check Imputation

**Why:** `Dispatch Date / Time` is missing on `73,405` rows (`14.45%`) — structurally missing for `99.06%` of 2016 records because CAD dispatch logging started on **2 April 2017** — and `29,165` rows have `Dispatch < Start` (negative reporting delay).
- **Fact Verification (Why Group-Minimum Imputation Was Rejected):** Below we empirically prove that taking `min(Dispatch Date / Time)` by `(Crime Name1, NIBRS Code)` returns **2017 dates**, and `min(positive_delay)` returns **`0.0` seconds**, both of which would fabricate unrealistic dispatch behaviour.
- **Decision (Option D):**
  1. Verify same-`Incident ID` backfill (`0` rows recoverable).
  2. **Main analysis:** Preserve missing `Dispatch Date / Time` as `NaT`, add `has_valid_dispatch_delay` (`Dispatch >= Start`), and compute `dispatch_delay_mins` strictly on valid non-negative delays.
  3. **Sensitivity check:** Create `dispatch_delay_mins_imputed` using the **group median delay** by `(Crime Name1, NIBRS Code, Start_Year)` applied **only where valid delays exist for that group**, plus flag `is_imputed_dispatch`.

In [8]:
# BEFORE evidence & proof of why group-minimum was rejected
raw_delay_mins = (df["Dispatch Date / Time"] - df["Start_Date_Time"]).dt.total_seconds() / 60.0
early_dispatch_mask = df["Dispatch Date / Time"].notna() & (raw_delay_mins < 0)
BASELINE["early_dispatch_rows"] = int(early_dispatch_mask.sum())

inc_disp_nunique = df.groupby("Incident ID")["Dispatch Date / Time"].transform("nunique")
recov_disp = int((df["Dispatch Date / Time"].isna() & (inc_disp_nunique == 1)).sum())

print(f"BEFORE — Missing Dispatch Date / Time                 : {int(df['Dispatch Date / Time'].isna().sum()):,} ({df['Dispatch Date / Time'].isna().mean()*100:.2f}%)")
print(f"BEFORE — Early Dispatch (Dispatch < Start)            : {int(early_dispatch_mask.sum()):,}")
print(f"CHECK  — Recoverable from same Incident ID            : {recov_disp:,}")

# Proof against group-minimum imputation
sample_min_ts = df.groupby(["Crime Name1", "NIBRS Code"])["Dispatch Date / Time"].min().head(3)
pos_delay_only = raw_delay_mins.where(raw_delay_mins >= 0)
print(f"PROOF  — Global min valid delay = {pos_delay_only.min():.2f} mins; sample group min timestamps:")
for idx, val in sample_min_ts.items():
    print(f"         {idx} -> {val}")

# Main analysis columns (strictly non-negative, no fabrication)
df["has_valid_dispatch_delay"] = df["Dispatch Date / Time"].notna() & (raw_delay_mins >= 0)
df["dispatch_delay_mins"] = np.where(df["has_valid_dispatch_delay"], raw_delay_mins, np.nan)

# Sensitivity check: group median by (Crime Name1, NIBRS Code, Start_Year) where valid delays exist
start_year_tmp = df["Start_Date_Time"].dt.year
grp_median_delay = (
    df.assign(_yr=start_year_tmp)
    .groupby(["Crime Name1", "NIBRS Code", "_yr"])["dispatch_delay_mins"]
    .transform("median")
)

can_impute_mask = df["Dispatch Date / Time"].isna() & grp_median_delay.notna()
df["is_imputed_dispatch"] = can_impute_mask
df["dispatch_delay_mins_imputed"] = df["dispatch_delay_mins"].copy()
df.loc[can_impute_mask, "dispatch_delay_mins_imputed"] = grp_median_delay[can_impute_mask]

# AFTER evidence: sensitivity comparison
main_valid = df["dispatch_delay_mins"].dropna()
sens_valid = df["dispatch_delay_mins_imputed"].dropna()
print(f"\nAFTER  — Main valid dispatch_delay_mins count         : {len(main_valid):,} | median={main_valid.median():.2f} mins | IQR={main_valid.quantile(0.75)-main_valid.quantile(0.25):.2f} mins")
print(f"AFTER  — Sensitivity imputed rows (is_imputed_dispatch): {int(df['is_imputed_dispatch'].sum()):,} (still NaN where group has no valid delays: {int(df['dispatch_delay_mins_imputed'].isna().sum()):,})")
print(f"AFTER  — Sensitivity combined count                   : {len(sens_valid):,} | median={sens_valid.median():.2f} mins | IQR={sens_valid.quantile(0.75)-sens_valid.quantile(0.25):.2f} mins")

BEFORE — Missing Dispatch Date / Time                 : 73,405 (14.45%)
BEFORE — Early Dispatch (Dispatch < Start)            : 29,165
CHECK  — Recoverable from same Incident ID            : 0
PROOF  — Global min valid delay = 0.00 mins; sample group min timestamps:
         ('Crime Against Not a Crime', '09C') -> 2018-07-23 19:54:44
         ('Crime Against Not a Crime', '90I') -> 2017-04-02 16:45:53
         ('Crime Against Person', '09A') -> 2017-04-29 01:25:38



AFTER  — Main valid dispatch_delay_mins count         : 405,536 | median=10.12 mins | IQR=1069.65 mins
AFTER  — Sensitivity imputed rows (is_imputed_dispatch): 63,943 (still NaN where group has no valid delays: 38,627)
AFTER  — Sensitivity combined count                   : 469,479 | median=12.60 mins | IQR=1196.80 mins


## Step 9 — [Item #5] Extract Calendar Features, `year_coverage`, `is_full_year` & `is_midnight_start`

**Why:** The dataset spans **1 July 2016 – 14 September 2026**, making **2016** (H2 only, `28,270` rows) and **2026** (Jan–Sep, `31,907` rows) partial calendar years. Plotting raw annual totals without flagging them creates false cliff-drops at both ends. Similarly, **`21,760` rows (`4.28%`)** start at exact `00:00:00`, creating an artificial midnight spike in time-of-day charts.  
**Decision (Option D):** Keep all rows, add `year_coverage`, `is_full_year` (`2017–2025`), and `is_midnight_start` for transparent filtering and sensitivity checks.

In [9]:
df["Start_Year"] = df["Start_Date_Time"].dt.year
df["Start_Month"] = df["Start_Date_Time"].dt.month
df["Start_DayOfWeek"] = df["Start_Date_Time"].dt.day_name()
df["Start_Hour"] = df["Start_Date_Time"].dt.hour

df["year_coverage"] = np.select(
    [df["Start_Year"] == 2016, df["Start_Year"] == 2026],
    ["Partial (H2 2016)", "Partial (Jan-Sep 2026)"],
    default="Full Year (2017-2025)",
)
df["is_full_year"] = df["Start_Year"].between(2017, 2025)
df["is_midnight_start"] = (df["Start_Date_Time"].dt.hour == 0) & (df["Start_Date_Time"].dt.minute == 0)

# Evidence table by year
yr_summary = (
    df.groupby(["Start_Year", "year_coverage", "is_full_year"])
    .agg(
        records=("Incident ID", "size"),
        min_date=("Start_Date_Time", "min"),
        max_date=("Start_Date_Time", "max"),
        midnight_starts=("is_midnight_start", "sum"),
    )
    .reset_index()
)
yr_summary["midnight_pct"] = (yr_summary["midnight_starts"] / yr_summary["records"] * 100).round(2)
print("=== YEAR COVERAGE & MIDNIGHT HEAPING EVIDENCE ===")
print(yr_summary.to_string(index=False))
print(f"\nTotal is_full_year == True : {int(df['is_full_year'].sum()):,} ({df['is_full_year'].mean()*100:.2f}%)")
print(f"Total is_midnight_start    : {int(df['is_midnight_start'].sum()):,} ({df['is_midnight_start'].mean()*100:.2f}%)")

=== YEAR COVERAGE & MIDNIGHT HEAPING EVIDENCE ===
 Start_Year          year_coverage  is_full_year  records   min_date            max_date  midnight_starts  midnight_pct
       2016      Partial (H2 2016)         False    28270 2016-07-01 2016-12-31 23:55:00             1022          3.62
       2017  Full Year (2017-2025)          True    56355 2017-01-01 2017-12-31 23:59:00             1930          3.42
       2018  Full Year (2017-2025)          True    54468 2018-01-01 2018-12-31 23:50:00             1778          3.26
       2019  Full Year (2017-2025)          True    52666 2019-01-01 2019-12-31 23:57:00             2002          3.80
       2020  Full Year (2017-2025)          True    45369 2020-01-01 2020-12-31 23:50:00             2348          5.18
       2021  Full Year (2017-2025)          True    44915 2021-01-01 2021-12-31 23:40:00             2653          5.91
       2022  Full Year (2017-2025)          True    48814 2022-01-01 2022-12-31 23:56:00             3055     

## Step 10 — [Item #6] Spatial Recovery of Police Districts (`geopandas`) & Fix ETL Slicing Bug

**Why:** An upstream ETL string-slicing bug corrupted every row where `Police District Number == 'Unknown'` (`1,557` rows): `"Unknown"[4:]` became `Beat == 'own'` and `"Unknown"[5]` became `Sector == 'w'` (lowercase), while `Police District Name` split inconsistently into `NaN` (`1,542`) and `'OTHER'` (`15`).  
**Decision (Option D):**
1. **Validation Gate:** Validate `montgomery_county_police_districts.geojson` against known valid-coord records first to prove polygon accuracy.
2. **Spatial Recovery:** Recover the exact `Police District Number` & `Police District Name` for all `15` `'OTHER'` rows that have valid coordinates (`has_valid_coords == True`), tracking provenance in `district_source`.
3. **Standardise Remainder (Option A):** Standardise the `1,542` Null Island rows (`Beat='own'` → `'Unknown'`, `Sector='w'` → `'Unknown'`, `Police District Name=NaN` → `'Unknown'`), plus edge cases (`Beat == '-PG'` (1), `PRA` nulls/`'1A3'`/`'000'`).

In [10]:
# BEFORE evidence
print("=== BEFORE POLICE DISTRICT / BEAT / SECTOR / PRA CLEANUP ===")
print(f"Police District Number == 'Unknown' : {int((df['Police District Number'] == 'Unknown').sum()):,}")
print(f"Police District Name NaN / 'OTHER'  : {int(df['Police District Name'].isna().sum()):,} / {int((df['Police District Name'] == 'OTHER').sum()):,}")
print(f"Beat == 'own' / '-PG'               : {int((df['Beat'] == 'own').sum()):,} / {int((df['Beat'] == '-PG').sum()):,}")
print(f"Sector == 'w'                       : {int((df['Sector'] == 'w').sum()):,}")
print(f"PRA NaN / '1A3' / '000'             : {int(df['PRA'].isna().sum()):,} / {int((df['PRA'] == '1A3').sum()):,} / {int((df['PRA'] == '000').sum()):,}")

# Step 10a: Validation gate on a 50,000-row sample of known valid-coord districts
known_dist_sample = df.loc[df["has_valid_coords"] & (df["Police District Number"] != "Unknown")].sample(50_000, random_state=42)
gdf_val_dist = gpd.GeoDataFrame(
    known_dist_sample[["Police District Number"]],
    geometry=gpd.points_from_xy(known_dist_sample["Longitude"], known_dist_sample["Latitude"]),
    crs="EPSG:4326",
)
val_dist_join = gpd.sjoin(gdf_val_dist, gdf_districts[["dist_num_poly", "geometry"]], how="left", predicate="within")
val_dist_join = val_dist_join[~val_dist_join.index.duplicated(keep="first")]
matched_val = val_dist_join["dist_num_poly"].notna()
agreement_dist = (val_dist_join.loc[matched_val, "Police District Number"] == val_dist_join.loc[matched_val, "dist_num_poly"]).mean() * 100
print(f"\nVALIDATION GATE — Police District polygon match rate: {matched_val.mean()*100:.2f}% | Agreement on matched: {agreement_dist:.2f}%")

# Step 10b: Spatial recovery for Unknown-district rows that have valid coordinates (15 rows)
df["district_source"] = np.where(df["Police District Number"] != "Unknown", "original", "unknown_no_coords")
unk_geo_mask = (df["Police District Number"] == "Unknown") & df["has_valid_coords"]

gdf_unk_dist = gpd.GeoDataFrame(
    df.loc[unk_geo_mask, []],
    geometry=gpd.points_from_xy(df.loc[unk_geo_mask, "Longitude"], df.loc[unk_geo_mask, "Latitude"]),
    crs="EPSG:4326",
)
unk_joined = gpd.sjoin(gdf_unk_dist, gdf_districts[["dist_num_poly", "dist_name_poly", "geometry"]], how="left", predicate="within")
unk_joined = unk_joined[~unk_joined.index.duplicated(keep="first")]

recovered_idx = unk_joined.dropna(subset=["dist_num_poly"]).index
df.loc[recovered_idx, "Police District Number"] = unk_joined.loc[recovered_idx, "dist_num_poly"]
df.loc[recovered_idx, "Police District Name"] = unk_joined.loc[recovered_idx, "dist_name_poly"]
df.loc[recovered_idx, "district_source"] = "spatial_imputed"

# Step 10c: Option A cleanup on remaining Unknown / corrupted ETL values
df["Police District Name"] = df["Police District Name"].fillna("Unknown").replace({"OTHER": "Unknown"})
df["Beat"] = df["Beat"].replace({"own": "Unknown", "-PG": "Unknown"})
df["Sector"] = df["Sector"].replace({"w": "Unknown"})
df["PRA"] = df["PRA"].fillna("Unknown").replace({"1A3": "Unknown", "000": "Unknown"})

# AFTER evidence
print("\n=== AFTER POLICE DISTRICT / BEAT / SECTOR / PRA CLEANUP ===")
print(f"Spatially recovered districts       : {len(recovered_idx):,} of {int(unk_geo_mask.sum()):,} valid-coord Unknown rows")
print(f"district_source counts              : {df['district_source'].value_counts().to_dict()}")
print(f"Police District Name NaN / 'OTHER'  : {int(df['Police District Name'].isna().sum()):,} / {int((df['Police District Name'] == 'OTHER').sum()):,}")
print(f"Beat == 'own' / '-PG'               : {int((df['Beat'] == 'own').sum()):,} / {int((df['Beat'] == '-PG').sum()):,}")
print(f"Sector == 'w'                       : {int((df['Sector'] == 'w').sum()):,}")
print(f"PRA NaN / '1A3' / '000'             : {int(df['PRA'].isna().sum()):,} / {int((df['PRA'] == '1A3').sum()):,} / {int((df['PRA'] == '000').sum()):,}")

=== BEFORE POLICE DISTRICT / BEAT / SECTOR / PRA CLEANUP ===
Police District Number == 'Unknown' : 1,557
Police District Name NaN / 'OTHER'  : 1,542 / 15
Beat == 'own' / '-PG'               : 1,557 / 1
Sector == 'w'                       : 1,557
PRA NaN / '1A3' / '000'             : 9 / 1 / 2



VALIDATION GATE — Police District polygon match rate: 98.96% | Agreement on matched: 95.44%



=== AFTER POLICE DISTRICT / BEAT / SECTOR / PRA CLEANUP ===
Spatially recovered districts       : 15 of 15 valid-coord Unknown rows
district_source counts              : {'original': 506549, 'unknown_no_coords': 1542, 'spatial_imputed': 15}
Police District Name NaN / 'OTHER'  : 0 / 0
Beat == 'own' / '-PG'               : 0 / 0
Sector == 'w'                       : 0
PRA NaN / '1A3' / '000'             : 0 / 0 / 0


## Step 11 — [Item #7] Spatial Verification & Cleaning of `State` (`State_Clean`, `state_source`)

**Why:** Raw `State` has `9,509` nulls plus `367` non-`MD` values including Maryland's numeric county code `'16'` (`224` rows), single digits (`'0'`, `'7'`, `'15'`, `'17'`), `'MC'` (1), and misentered border codes (`DC`, `VA`, etc.).  
**Decision (Option D):** Preserve raw `State`, create `State_Clean` and `state_source`: first verify all valid-coordinate missing/mislabeled rows via exact `geopandas` point-in-polygon against the official Maryland boundary (`maryland_zip_codes.geojson` / `montgomery_county_police_districts.geojson`), then apply Option A (jurisdiction default `'MD'` for remaining Null Island missing/corrupted rows).

In [11]:
# BEFORE evidence
print("=== BEFORE STATE CLEANING ===")
print(df["State"].value_counts(dropna=False).to_string())

INVALID_STATE_TOKENS = {"16", "7", "17", "15", "0", "MC"}
df["State_Clean"] = df["State"].copy()
df["state_source"] = np.where(df["State"] == "MD", "original_md", "other_or_missing")

# Spatial point-in-polygon check for all missing or non-MD rows that have valid coordinates
state_fix_candidates = (df["State"].isna() | (df["State"] != "MD")) & df["has_valid_coords"]
gdf_state_pts = gpd.GeoDataFrame(
    df.loc[state_fix_candidates, []],
    geometry=gpd.points_from_xy(df.loc[state_fix_candidates, "Longitude"], df.loc[state_fix_candidates, "Latitude"]),
    crs="EPSG:4326",
)
# Check containment inside official Montgomery County / Maryland polygons
in_md_join = gpd.sjoin(gdf_state_pts, gdf_districts[["AREA_P", "geometry"]], how="left", predicate="within")
in_md_join = in_md_join[~in_md_join.index.duplicated(keep="first")]
spatial_md_idx = in_md_join.dropna(subset=["AREA_P"]).index

df.loc[spatial_md_idx, "State_Clean"] = "MD"
df.loc[spatial_md_idx, "state_source"] = "spatial_imputed_md"

# Option A: fill remaining nulls & corrupted numeric/MC codes with 'MD' (Montgomery County jurisdiction)
rem_fix_mask = df["State_Clean"].isna() | df["State_Clean"].isin(INVALID_STATE_TOKENS)
df.loc[rem_fix_mask, "State_Clean"] = "MD"
df.loc[rem_fix_mask, "state_source"] = "jurisdiction_default_md"
df.loc[(df["State_Clean"] != "MD") & (df["state_source"] == "other_or_missing"), "state_source"] = "original_border_state"

# AFTER evidence
print("\n=== AFTER STATE CLEANING ===")
print("State_Clean value counts:")
print(df["State_Clean"].value_counts(dropna=False).to_string())
print("\nstate_source provenance counts:")
print(df["state_source"].value_counts().to_string())

=== BEFORE STATE CLEANING ===
State
MD     498230
NaN      9509
16        224
DC        121
VA         12
ME          2
MS          2
7           1
ND          1
17          1
MC          1
0           1
15          1



=== AFTER STATE CLEANING ===
State_Clean value counts:
State_Clean
MD    507976
DC       118
VA        12

state_source provenance counts:
state_source
original_md                498230
spatial_imputed_md           9051
jurisdiction_default_md       695
original_border_state         130


## Step 12 — [Item #8] Spatial Validation & Imputation of `Zip Code` (`Zip_Code_Clean`, `zip_source`, `is_md_zip`)

**Why:** Raw `Zip Code` is cast as `float64` due to `3,474` nulls and contains `19` malformed non-5-digit numbers (`< 10000`, e.g., `6.0`, `209.0`, `2902.0`).  
**Decision (Option D):**
1. Preserve raw `Zip Code`; create `Zip_Code_Clean` as a 5-digit string and set the `19` values `< 10000` to `NaN`.
2. **Validation Gate:** Run an exact `geopandas` point-in-polygon join (`maryland_zip_codes.geojson`) on a 50,000-row sample of known valid-zip records and verify high agreement before imputing.
3. Impute **only** missing/invalid `Zip_Code_Clean` rows that have valid coordinates (`has_valid_coords == True`), keeping all `508,106` rows in the dataset (Null Island rows with missing zips remain `NaN`).
4. Add `zip_source` (`original`, `spatial_imputed`, `missing_no_coords`, `unmatched_border`) and `is_md_zip` (`20600–21999`).

In [12]:
# Step 12a: Format valid 5-digit zips cleanly into Zip_Code_Clean without overwriting raw Zip Code
zip_num = df["Zip Code"]
valid_5digit_mask = zip_num.notna() & (zip_num >= 10000) & (zip_num <= 99999)
malformed_zip_mask = zip_num.notna() & ~valid_5digit_mask

df["Zip_Code_Clean"] = pd.Series(pd.NA, index=df.index, dtype="object")
df.loc[valid_5digit_mask, "Zip_Code_Clean"] = zip_num.loc[valid_5digit_mask].astype("int64").astype(str).str.zfill(5)
df["zip_source"] = np.where(valid_5digit_mask, "original", "missing_or_invalid")

print("=== BEFORE ZIP SPATIAL IMPUTATION ===")
print(f"Valid original 5-digit zips         : {int(valid_5digit_mask.sum()):,}")
print(f"Malformed (<10000) set to NaN       : {int(malformed_zip_mask.sum()):,}")
print(f"Total missing/invalid to resolve    : {int(df['Zip_Code_Clean'].isna().sum()):,}")

# Step 12b: Validation Gate on 50,000 known valid MD zips with valid coordinates
val_zip_sample = df.loc[valid_5digit_mask & df["has_valid_coords"]].sample(50_000, random_state=42)
gdf_val_zip = gpd.GeoDataFrame(
    val_zip_sample[["Zip_Code_Clean"]],
    geometry=gpd.points_from_xy(val_zip_sample["Longitude"], val_zip_sample["Latitude"]),
    crs="EPSG:4326",
)
val_zip_join = gpd.sjoin(gdf_val_zip, gdf_zips[["zip_poly", "geometry"]], how="left", predicate="within")
val_zip_join = val_zip_join[~val_zip_join.index.duplicated(keep="first")]
matched_z = val_zip_join["zip_poly"].notna()
zip_agreement = (val_zip_join.loc[matched_z, "Zip_Code_Clean"] == val_zip_join.loc[matched_z, "zip_poly"]).mean() * 100
print(f"\nVALIDATION GATE — MD ZIP polygon match rate: {matched_z.mean()*100:.2f}% | Agreement on matched: {zip_agreement:.2f}%")
assert zip_agreement >= 90.0, f"ZIP spatial join validation agreement too low: {zip_agreement:.2f}%"

# Step 12c: Impute missing/invalid Zip_Code_Clean ONLY where valid coordinates exist
impute_zip_mask = df["Zip_Code_Clean"].isna() & df["has_valid_coords"]
gdf_imp_zip = gpd.GeoDataFrame(
    df.loc[impute_zip_mask, []],
    geometry=gpd.points_from_xy(df.loc[impute_zip_mask, "Longitude"], df.loc[impute_zip_mask, "Latitude"]),
    crs="EPSG:4326",
)
imp_zip_join = gpd.sjoin(gdf_imp_zip, gdf_zips[["zip_poly", "geometry"]], how="left", predicate="within")
imp_zip_join = imp_zip_join[~imp_zip_join.index.duplicated(keep="first")]

filled_zip_idx = imp_zip_join.dropna(subset=["zip_poly"]).index
df.loc[filled_zip_idx, "Zip_Code_Clean"] = imp_zip_join.loc[filled_zip_idx, "zip_poly"]
df.loc[filled_zip_idx, "zip_source"] = "spatial_imputed"

# Distinguish remaining NaNs by whether coordinates were missing vs outside MD polygons
df.loc[df["Zip_Code_Clean"].isna() & ~df["has_valid_coords"], "zip_source"] = "missing_no_coords"
df.loc[df["Zip_Code_Clean"].isna() & df["has_valid_coords"], "zip_source"] = "unmatched_outside_md_poly"

# Flag Maryland ZIP range (20600-21999)
zip_int_clean = pd.to_numeric(df["Zip_Code_Clean"], errors="coerce")
df["is_md_zip"] = zip_int_clean.between(20600, 21999)

# AFTER evidence
print("\n=== AFTER ZIP SPATIAL IMPUTATION ===")
print(f"Spatially imputed zips              : {len(filled_zip_idx):,} of {int(impute_zip_mask.sum()):,} valid-coord missing/invalid rows")
print(f"Remaining NaN zips                  : {int(df['Zip_Code_Clean'].isna().sum()):,} ({df['Zip_Code_Clean'].isna().mean()*100:.3f}%)")
print(f"zip_source breakdown                : {df['zip_source'].value_counts().to_dict()}")
print(f"is_md_zip == True                   : {int(df['is_md_zip'].sum()):,} ({df['is_md_zip'].mean()*100:.2f}%)")

=== BEFORE ZIP SPATIAL IMPUTATION ===
Valid original 5-digit zips         : 504,613
Malformed (<10000) set to NaN       : 19
Total missing/invalid to resolve    : 3,493



VALIDATION GATE — MD ZIP polygon match rate: 99.72% | Agreement on matched: 90.94%



=== AFTER ZIP SPATIAL IMPUTATION ===
Spatially imputed zips              : 3,391 of 3,399 valid-coord missing/invalid rows
Remaining NaN zips                  : 102 (0.020%)
zip_source breakdown                : {'original': 504613, 'spatial_imputed': 3391, 'missing_no_coords': 94, 'unmatched_outside_md_poly': 8}
is_md_zip == True                   : 507,874 (99.95%)


## Step 13 — [Item #9] Standardise `City` (`City_Clean` with Full Fuzzy-Match Audit)

**Why:** Raw `City` has **`314` unique values**, of which **`236` (`75.2%`) appear ≤ 5 times** — heavily polluted by abbreviations (`TP`: 538, `MCG`: 387), misspellings (`SILVE4R SPRING`, `RCKVILLE`, `GAISTHERSBURG`), punctuation (`ROCKVILLE,`), and invalid entries (`'0'`, `'MARYLAND'`).  
**Decision (Option D):** Preserve raw `City`, create `City_Clean` in 4 audited stages (logging unique counts at each stage):
1. Uppercase, trim, collapse repeated spaces, strip punctuation.
2. Map known abbreviations (`TP` → `TAKOMA PARK`, `N BETHESDA` → `NORTH BETHESDA`, etc.) and treat non-city entries (`MCG`, `0`, `MARYLAND`) as `'NON-CITY/UNKNOWN'`; leave the `37` nulls as `NaN`.
3. Apply `difflib.get_close_matches(cutoff=0.85)` against the canonical Montgomery County / DMV community list and print **every single fuzzy match** for manual audit.
4. Group only remaining unmatched entries with `count <= 5` into `'OTHER/UNKNOWN'`.

In [13]:
# BEFORE evidence
raw_city_nunique = df["City"].nunique(dropna=True)
print(f"BEFORE — Raw unique cities: {raw_city_nunique} | Nulls: {int(df['City'].isna().sum())}")

# Stage 1: Basic text normalisation (keep NaN as NaN)
city_clean = df["City"].copy()
non_null_c = city_clean.notna()
city_clean.loc[non_null_c] = (
    city_clean.loc[non_null_c]
    .astype(str)
    .str.upper()
    .str.strip()
    .str.replace(r"[^A-Z0-9\s]", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
print(f"Stage 1 (Upper/Strip/Punct) -> unique cities: {city_clean.nunique(dropna=True)}")

# Stage 2: Explicit curated abbreviation, typo & non-city dictionary
CITY_EXPLICIT_MAP = {
    "TP": "TAKOMA PARK",
    "TAKOMA": "TAKOMA PARK",
    "N BETHESDA": "NORTH BETHESDA",
    "N BETHESDAQ": "NORTH BETHESDA",
    "NORTH BEHTESDA": "NORTH BETHESDA",
    "NORTH BETHSDA": "NORTH BETHESDA",
    "N POTOMAC": "NORTH POTOMAC",
    "MONT VILLAGE": "MONTGOMERY VILLAGE",
    "MONTGOMERY VILL": "MONTGOMERY VILLAGE",
    "SS": "SILVER SPRING",
    "SILV ER SPRING": "SILVER SPRING",
    "SILVE4R SPRING": "SILVER SPRING",
    "GATHERSBURG": "GAITHERSBURG",
    "GAITHERBURG": "GAITHERSBURG",
    "RCKVILLE": "ROCKVILLE",
    "GERANTOWN": "GERMANTOWN",
    "CERWOOD": "DERWOOD",
    "GARETT PARK": "GARRETT PARK",
    "WASH": "WASHINGTON",
    "WASHINGTON DC": "WASHINGTON",
    "DC": "WASHINGTON",
    # Non-city / county / invalid entries
    "MCG": "NON-CITY/UNKNOWN",
    "MONTGOMERY COUNTY": "NON-CITY/UNKNOWN",
    "0": "NON-CITY/UNKNOWN",
    "MARYLAND": "NON-CITY/UNKNOWN",
    "MD": "NON-CITY/UNKNOWN",
}
city_clean = city_clean.replace(CITY_EXPLICIT_MAP)
print(f"Stage 2 (Curated Dict)      -> unique cities: {city_clean.nunique(dropna=True)}")

# Stage 3: High-threshold fuzzy matching (difflib >= 0.85) against canonical places (count >= 20)
# Manual audit protection: distinct real Montgomery County communities that must NEVER be fuzzy-merged
PROTECTED_REAL_PLACES = {"COLESVILLE"}  # Distinct East County community (prevent false merge into POOLESVILLE)

city_counts = city_clean.value_counts()
canonical_cities = sorted([c for c in city_counts[city_counts >= 20].index if c != "NON-CITY/UNKNOWN"])

fuzzy_candidates = [c for c in city_counts[city_counts < 20].index if c not in {"NON-CITY/UNKNOWN"} | PROTECTED_REAL_PLACES]
fuzzy_map = {}
for cand in fuzzy_candidates:
    matches = difflib.get_close_matches(cand, canonical_cities, n=1, cutoff=0.85)
    if matches and cand != matches[0]:
        fuzzy_map[cand] = matches[0]

print(f"\n=== MANUAL AUDIT OF FUZZY MATCHES ({len(fuzzy_map)} verified pairs, cutoff=0.85; protected: {sorted(PROTECTED_REAL_PLACES)}) ===")
for k, v in sorted(fuzzy_map.items()):
    print(f"  {k:28s} (n={int(city_counts[k]):2d}) -> {v}")

city_clean = city_clean.replace(fuzzy_map)
print(f"\nStage 3 (Fuzzy Match >=0.85)-> unique cities: {city_clean.nunique(dropna=True)}")

# Stage 4: Group remaining unmatched rare entries (count <= 5) into OTHER/UNKNOWN
post_fuzzy_counts = city_clean.value_counts()
rare_unmatched = post_fuzzy_counts[(post_fuzzy_counts <= 5) & (post_fuzzy_counts.index != "NON-CITY/UNKNOWN")].index
city_clean = city_clean.replace({c: "OTHER/UNKNOWN" for c in rare_unmatched})

df["City_Clean"] = city_clean

# AFTER evidence
print(f"Stage 4 (Rare <=5 grouped)  -> unique cities: {df['City_Clean'].nunique(dropna=True)} (reduced from {raw_city_nunique})")
print(f"Nulls preserved             : {int(df['City_Clean'].isna().sum())} (raw was {int(df['City'].isna().sum())})")
print("\nTop 15 City_Clean values:")
print(df["City_Clean"].value_counts(dropna=False).head(15).to_string())

BEFORE — Raw unique cities: 314 | Nulls: 37


Stage 1 (Upper/Strip/Punct) -> unique cities: 304


Stage 2 (Curated Dict)      -> unique cities: 283

=== MANUAL AUDIT OF FUZZY MATCHES (159 verified pairs, cutoff=0.85; protected: ['COLESVILLE']) ===
  APENCERVILLE                 (n= 1) -> SPENCERVILLE
  BARNESVIILE                  (n= 1) -> BARNESVILLE
  BARNSVILLE                   (n= 4) -> BARNESVILLE
  BEALSVILLE                   (n= 1) -> BEALLSVILLE
  BEHESDA                      (n= 4) -> BETHESDA
  BEHTESDA                     (n= 3) -> BETHESDA
  BELTSVILLE PG                (n= 3) -> BELTSVILLE
  BELTVILLE                    (n= 1) -> BELTSVILLE
  BETEHSDA                     (n= 1) -> BETHESDA
  BETESDA                      (n= 2) -> BETHESDA
  BETHEDA                      (n= 5) -> BETHESDA
  BETHESA                      (n= 3) -> BETHESDA
  BETHESDAS                    (n= 1) -> BETHESDA
  BETHSDA                      (n= 2) -> BETHESDA
  BROOKVILLE                   (n=11) -> BROOKEVILLE
  BURTONSVILE                  (n= 1) -> BURTONSVILLE
  BURTOSNVILLE            


Stage 3 (Fuzzy Match >=0.85)-> unique cities: 124


Stage 4 (Rare <=5 grouped)  -> unique cities: 55 (reduced from 314)
Nulls preserved             : 37 (raw was 37)

Top 15 City_Clean values:
City_Clean
SILVER SPRING         175715
GAITHERSBURG           71736
ROCKVILLE              71479
GERMANTOWN             48632
BETHESDA               36164
TAKOMA PARK            17174
MONTGOMERY VILLAGE     14914
CHEVY CHASE            10074
POTOMAC                10049
DERWOOD                 8859
KENSINGTON              7935
CLARKSBURG              7327
OLNEY                   7128
BURTONSVILLE            6314
DAMASCUS                4125


## Step 14 — [Item #10] Standardise `Place` (`Place_Clean`, `Place_Group`, `is_legacy_place`)

**Why:** Raw `Place` has `100` granular labels, inconsistent hyphen spacing (`Residence -Townhouse/Duplex`: `22,325` rows; `Shelter-Mission/Homeless`: `1,022` rows), and a deprecated legacy label (`School/College - DO NOT USE`: `6,433` rows) that cannot safely be guessed into K-12 vs. College.  
**Decision (Option D):** Preserve raw `Place`, normalise hyphen spacing in `Place_Clean`, rename legacy school label to `'School/College (Legacy)'` with `is_legacy_place` flag, and map 100% of rows into clean analytical `Place_Group` categories.

In [14]:
# BEFORE evidence: inspect legacy school counts by year & hyphen anomalies
legacy_mask = df["Place"] == "School/College - DO NOT USE"
tight_hyphen_before = df["Place"].str.contains(r"[A-Za-z]-[A-Za-z]| -[A-Za-z]|[A-Za-z]- ", regex=True).sum()
print(f"BEFORE — Unique Place labels                    : {df['Place'].nunique()}")
print(f"BEFORE — Tight/inconsistent hyphen rows         : {int(tight_hyphen_before):,}")
print(f"BEFORE — 'School/College - DO NOT USE' rows     : {int(legacy_mask.sum()):,}")
print("         Legacy school by year:", df.loc[legacy_mask, "Start_Year"].value_counts().sort_index().to_dict())

# Create Place_Clean and is_legacy_place
df["is_legacy_place"] = legacy_mask
df["Place_Clean"] = (
    df["Place"]
    .str.replace(r"\s*-\s*", " - ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .replace({"School/College - DO NOT USE": "School/College (Legacy)"})
)

# Deterministic rule-based grouping covering 100% of Place_Clean values
def assign_place_group(val: str) -> str:
    v = val.lower()
    if v == "other/unknown":
        return "Other / Unknown"
    if v.startswith("residence") or "nursing home" in v or "shelter" in v or "hotel" in v or "motel" in v or "assisted living" in v:
        return "Residence & Lodging"
    if v.startswith("street") or "highway" in v or "alley" in v or "sidewalk" in v or "bridge" in v:
        return "Street & Roadway"
    if "parking" in v or "garage" in v:
        return "Parking Lot & Garage"
    if "school" in v or "college" in v or "daycare" in v or "child care" in v:
        return "School & Education"
    if v.startswith("retail") or "store" in v or "grocery" in v or "supermarket" in v or "mall" in v or "pharmacy" in v or "gas station" in v or "auto dealership" in v or "laundromat" in v or "barber" in v or "pawn" in v or "rental" in v:
        return "Retail & Commercial"
    if "restaurant" in v or "bar" in v or "nightclub" in v or "liquor" in v or "fast food" in v:
        return "Food & Nightlife"
    if "bank" in v or "atm" in v or "commercial" in v or "office" in v or "construction" in v or "warehouse" in v or "industrial" in v or "storage" in v:
        return "Office, Financial & Industrial"
    if "park" in v or "recreation" in v or "field" in v or "woods" in v or "lake" in v or "pool" in v or "golf" in v or "camp" in v or "arena" in v or "amusement" in v or "gambling" in v or "theater" in v or "community center" in v:
        return "Park & Recreation"
    if "government" in v or "police" in v or "jail" in v or "prison" in v or "hospital" in v or "medical" in v or "library" in v or "military" in v or "post office" in v or "fire" in v:
        return "Government, Medical & Civic"
    if "metro" in v or "bus" in v or "train" in v or "rail" in v or "air" in v or "dock" in v or "terminal" in v or "rest area" in v or "transit" in v:
        return "Transit & Transportation"
    if "church" in v or "synagogue" in v or "temple" in v or "mosque" in v or "religious" in v or "cemetery" in v:
        return "Religious & Community"
    if "cyberspace" in v or "internet" in v:
        return "Cyberspace"
    return "Other / Miscellaneous"

df["Place_Group"] = df["Place_Clean"].map(assign_place_group)
assert df["Place_Group"].isna().sum() == 0, "Found unmapped Place_Group rows!"

# AFTER evidence
print("\n=== AFTER PLACE CLEANING ===")
print(f"AFTER  — Place_Clean unique labels              : {df['Place_Clean'].nunique()}")
print(f"AFTER  — Place_Group categories                 : {df['Place_Group'].nunique()} (0 nulls)")
print("\nPlace_Group distribution:")
print(df["Place_Group"].value_counts().to_string())

BEFORE — Unique Place labels                    : 100
BEFORE — Tight/inconsistent hyphen rows         : 23,347
BEFORE — 'School/College - DO NOT USE' rows     : 6,433
         Legacy school by year: {2016: 619, 2017: 1665, 2018: 1619, 2019: 1611, 2020: 719, 2021: 158, 2022: 11, 2023: 12, 2024: 8, 2025: 6, 2026: 5}



=== AFTER PLACE CLEANING ===
AFTER  — Place_Clean unique labels              : 100
AFTER  — Place_Group categories                 : 14 (0 nulls)

Place_Group distribution:
Place_Group
Residence & Lodging               162997
Street & Roadway                  106287
Retail & Commercial                70140
Parking Lot & Garage               67227
Other / Unknown                    40895
School & Education                 15250
Office, Financial & Industrial     11798
Food & Nightlife                   11015
Government, Medical & Civic        10281
Park & Recreation                   5764
Transit & Transportation            2205
Cyberspace                          2053
Religious & Community               1207
Other / Miscellaneous                987


## Step 15 — [Items #11 & #12] Add NIBRS Taxonomy Flags & Verify Offence-Level Grain

**Why:** Under FBI NIBRS rules, `Crime Name1 == 'Crime Against Not a Crime'` (`6,158` rows: Runaway & Justifiable Homicide) and `NIBRS Code == '90Z'` (`121,221` rows: `All Other Offenses`, including Police Information & Mental Illness Emergency Petitions) are legally distinct from core Part I/Group A crimes. Meanwhile, `40,505` rows represent secondary offences sharing an `Incident ID`.  
**Decision (Option A on both):** Retain all `508,106` offence-level rows, add explicit flags `is_not_a_crime` and `is_90z_other`, and verify incident-level counting uses `Incident ID` (`467,601` unique, never `CR Number`).

In [15]:
df["is_not_a_crime"] = df["Crime Name1"] == "Crime Against Not a Crime"
df["is_90z_other"] = df["NIBRS Code"] == "90Z"

print("=== NIBRS TAXONOMY & GRAIN VERIFICATION ===")
print(f"Total offence rows (grain preserved) : {len(df):,}")
print(f"Unique Incident IDs (true incidents) : {df['Incident ID'].nunique():,}")
print(f"Unique CR Numbers (67 shared cases)  : {df['CR Number'].nunique():,}")
print(f"is_not_a_crime == True               : {int(df['is_not_a_crime'].sum()):,} ({df['is_not_a_crime'].mean()*100:.2f}%)")
print(f"is_90z_other == True                 : {int(df['is_90z_other'].sum()):,} ({df['is_90z_other'].mean()*100:.2f}%)")
print(f"Core criminal offences (neither flag): {int((~df['is_not_a_crime'] & ~df['is_90z_other']).sum()):,} ({(~df['is_not_a_crime'] & ~df['is_90z_other']).mean()*100:.2f}%)")

=== NIBRS TAXONOMY & GRAIN VERIFICATION ===
Total offence rows (grain preserved) : 508,106
Unique Incident IDs (true incidents) : 467,601
Unique CR Numbers (67 shared cases)  : 467,534
is_not_a_crime == True               : 6,158 (1.21%)
is_90z_other == True                 : 121,221 (23.86%)
Core criminal offences (neither flag): 380,727 (74.93%)


## Step 16 — [Item #13] Drop Redundant/Sparse Address Columns & Clean `Block Address` / `Address Number`

**Why:**
- `Location` is 100% redundant with `Latitude`/`Longitude` and has 4 leading spaces on every row (`508,106`).
- `Street Prefix` (`95.56%` null) and `Street Suffix` (`98.64%` null) are structurally empty.
- `Block Address` contains double spaces on `451,976` rows (`1000 BLK  MILESTONE DR`).
- `Address Number` is cast as `float64` and contains `1` zero (`'O00 BLK'` typo) and `184` extreme values (`> 50,000`, up to `2,090,600`).  
**Decision (Option A):** Drop `Location`, `Street Prefix`, `Street Suffix`; collapse double spaces in `Block Address`; cast `Address Number` to nullable `Int64` and nullify `== 0` or `> 50,000` with `has_invalid_addr_num`.

In [16]:
# BEFORE evidence
cols_to_drop = ["Location", "Street Prefix", "Street Suffix"]
dbl_space_before = int(df["Block Address"].dropna().str.contains(r"  ", regex=True).sum())
invalid_addr_mask = df["Address Number"].notna() & ((df["Address Number"] == 0) | (df["Address Number"] > 50000))

print(f"BEFORE — Columns count                        : {df.shape[1]}")
print(f"BEFORE — Block Address rows with double space : {dbl_space_before:,}")
print(f"BEFORE — Address Number == 0 or > 50,000      : {int(invalid_addr_mask.sum()):,} (dtype={df['Address Number'].dtype})")

# Apply Option A
df = df.drop(columns=cols_to_drop)
non_null_blk = df["Block Address"].notna()
df.loc[non_null_blk, "Block Address"] = (
    df.loc[non_null_blk, "Block Address"].str.strip().str.replace(r"\s{2,}", " ", regex=True)
)

df["has_invalid_addr_num"] = invalid_addr_mask
df.loc[invalid_addr_mask, "Address Number"] = np.nan
df["Address Number"] = df["Address Number"].astype("Int64")

# AFTER evidence
dbl_space_after = int(df["Block Address"].dropna().str.contains(r"  ", regex=True).sum())
print(f"\nAFTER  — Dropped columns                      : {cols_to_drop}")
print(f"AFTER  — Block Address rows with double space : {dbl_space_after:,}")
print(f"AFTER  — Address Number dtype                 : {df['Address Number'].dtype} | min={df['Address Number'].min()} | max={df['Address Number'].max()}")

BEFORE — Columns count                        : 56
BEFORE — Block Address rows with double space : 451,976
BEFORE — Address Number == 0 or > 50,000      : 185 (dtype=float64)



AFTER  — Dropped columns                      : ['Location', 'Street Prefix', 'Street Suffix']
AFTER  — Block Address rows with double space : 0
AFTER  — Address Number dtype                 : Int64 | min=1 | max=49200


## Step 17 — [Item #14] Clean `Agency` (`Agency_Clean`) & Flag `Victims == 22` Outlier (`is_victim_outlier`)

**Why:** `Agency` contains `3` cryptic `'P'` rows alongside `4` `'OTHR'` rows; inspecting them confirms none indicate a named municipal department. `Victims` has `1` extreme right-tail value (`22` victims in a single Simple Assault at an Apartment/Condo), which is plausible and should **not** be deleted or winsorised, but must be flagged for sensitivity reporting.  
**Decision (Option D):** Preserve raw `Agency` and `Victims`, create `Agency_Clean` (`'P'` → `'OTHR'`), add `is_victim_outlier` (`Victims >= 10`), and prove robustness of `Victims` summary statistics with vs. without the flagged row.

In [17]:
# BEFORE evidence
print("=== INSPECTING CRYPTIC AGENCY == 'P' ROWS ===")
print(df.loc[df["Agency"] == "P", ["Incident ID", "Agency", "Police District Name", "City_Clean", "Crime Name2"]].to_string())

print("\n=== INSPECTING VICTIMS >= 10 OUTLIER ===")
print(df.loc[df["Victims"] >= 10, ["Incident ID", "Start_Date_Time", "Victims", "Crime Name1", "Crime Name2", "Place_Clean"]].to_string())

# Apply Option D
df["Agency_Clean"] = df["Agency"].replace({"P": "OTHR"})
df["is_victim_outlier"] = df["Victims"] >= 10

# AFTER evidence & Victims sensitivity check
v_all = df["Victims"]
v_no_out = df.loc[~df["is_victim_outlier"], "Victims"]
print("\n=== AFTER AGENCY & VICTIMS CHECK ===")
print("Agency_Clean counts:", df["Agency_Clean"].value_counts().to_dict())
print(f"Victims (ALL rows)       : mean={v_all.mean():.5f} | median={v_all.median():.1f} | max={v_all.max()}")
print(f"Victims (excl. outlier)  : mean={v_no_out.mean():.5f} | median={v_no_out.median():.1f} | max={v_no_out.max()}")
print(f"Mean delta from outlier  : {abs(v_all.mean() - v_no_out.mean()):.6f} (conclusions 100% robust)")

=== INSPECTING CRYPTIC AGENCY == 'P' ROWS ===
        Incident ID Agency Police District Name          City_Clean               Crime Name2
95707     201494823      P   MONTGOMERY VILLAGE             DERWOOD        All Other Offenses
375698    201214930      P              WHEATON           ROCKVILLE        All Other Offenses
447636    201145776      P   MONTGOMERY VILLAGE  MONTGOMERY VILLAGE  Theft From Motor Vehicle

=== INSPECTING VICTIMS >= 10 OUTLIER ===
        Incident ID     Start_Date_Time  Victims           Crime Name1     Crime Name2                  Place_Clean
504568    201090564 2016-07-23 16:30:00       22  Crime Against Person  Simple Assault  Residence - Apartment/Condo

=== AFTER AGENCY & VICTIMS CHECK ===
Agency_Clean counts: {'MCPD': 434446, 'RCPD': 28878, 'GPD': 28541, 'TPPD': 14841, 'MCFM': 1237, 'MCSO': 156, 'OTHR': 7}
Victims (ALL rows)       : mean=1.02183 | median=1.0 | max=22
Victims (excl. outlier)  : mean=1.02179 | median=1.0 | max=9
Mean delta from outlier

## Step 18 — Comprehensive Before-vs-After Executive Comparison & 12 Automated Assertions

**Why:** Prove end-to-end that every single data quality defect documented in `Dataset_Quality_Analysis.md` has been resolved, zero rows were accidentally dropped, and every structural invariant holds before exporting the frozen dataset.

In [18]:
comparison = pd.DataFrame([
    ("Total Rows (Zero Accidental Loss)", f"{BASELINE['rows']:,}", f"{len(df):,}", "Preserved 100%"),
    ("Total Columns", f"{BASELINE['columns']}", f"{df.shape[1]} (27 raw + {df.shape[1]-27} clean/flags)", f"3 dropped, {df.shape[1]-27} engineered"),
    ("Duplicate (Incident ID, Offence Code)", f"{BASELINE['dup_incident_offence_key']:,}", f"{int(df.duplicated(subset=['Incident ID', 'Offence Code']).sum()):,}", "Primary key intact"),
    ("Null Island Coordinates (0.0, 0.0)", f"{BASELINE['null_island_coords']:,}", f"{int(((df['Latitude']==0)|(df['Longitude']==0)).sum()):,}", "Converted to NaN + flag"),
    ("Negative Durations (Start > End)", f"{BASELINE['negative_durations']:,}", f"{int((df['duration_mins'] < 0).sum()):,}", "Nullified + flagged"),
    ("Negative Dispatch Delays (Main Col)", f"{BASELINE['early_dispatch_rows']:,}", f"{int((df['dispatch_delay_mins'] < 0).sum()):,}", "Excluded via flag"),
    ("Corrupted Beat ('own') / Sector ('w')", f"{BASELINE['corrupted_beat_own']:,} / {BASELINE['corrupted_sector_w']:,}", f"{int((df['Beat']=='own').sum()):,} / {int((df['Sector']=='w').sum()):,}", "15 spatial + 1,542 Unknown"),
    ("Police District Name NaN / 'OTHER'", f"1,542 / 15", f"{int(df['Police District Name'].isna().sum()):,} / {int((df['Police District Name']=='OTHER').sum()):,}", "100% standardised"),
    ("Missing or Non-MD State (State_Clean)", f"{BASELINE['missing_or_non_md_state']:,}", f"{int((df['State_Clean'].isna() | (df['State_Clean']!='MD')).sum()):,}", "Verified via GeoPandas"),
    ("Missing or Invalid Zip (Zip_Code_Clean)", f"{BASELINE['missing_or_invalid_zip']:,}", f"{int(df['Zip_Code_Clean'].isna().sum()):,}", "Imputed via MD polygons"),
    ("Unique Cities (City -> City_Clean)", f"{BASELINE['unique_cities']:,}", f"{df['City_Clean'].nunique(dropna=True):,}", "Typos & abbrevs resolved"),
    ("Deprecated Place Label ('DO NOT USE')", f"{BASELINE['legacy_place_rows']:,}", f"{int((df['Place_Clean']=='School/College - DO NOT USE').sum()):,}", "Renamed + Place_Group"),
    ("Block Address Double Spaces", f"{BASELINE['double_space_block_addr']:,}", f"{int(df['Block Address'].dropna().str.contains(r'  ', regex=True).sum()):,}", "100% collapsed"),
    ("Invalid Address Number (0 or >50,000)", f"{BASELINE['invalid_addr_num']:,}", f"{int(((df['Address Number']==0)|(df['Address Number']>50000)).sum()):,}", "Nullified + Int64"),
    ("Cryptic Agency == 'P' (Agency_Clean)", f"{BASELINE['cryptic_agency_p']:,}", f"{int((df['Agency_Clean']=='P').sum()):,}", "Merged into OTHR"),
], columns=["Quality Metric", "BEFORE (Raw)", "AFTER (Cleaned)", "Verification Result"])

print("=== EXECUTIVE BEFORE vs AFTER COMPARISON ===")
print(comparison.to_string(index=False))

# 12 Automated Verification Assertions (aborts export if any check fails)
assert len(df) == 508_106, "Row count changed!"
assert df.duplicated(subset=["Incident ID", "Offence Code"]).sum() == 0, "Primary key broken!"
assert ((df["Latitude"] == 0) | (df["Longitude"] == 0)).sum() == 0, "Null Island coords remain!"
assert df.loc[df["has_valid_coords"], "Latitude"].between(38.90, 39.38).all(), "Out-of-MoCo Lat!"
assert (df["End_Date_Time"] < df["Start_Date_Time"]).sum() == 0, "Negative End < Start remains!"
assert (df["duration_mins"] < 0).sum() == 0 and (df["dispatch_delay_mins"] < 0).sum() == 0, "Negative duration/delay!"
assert df["Beat"].isin(["own", "-PG"]).sum() == 0 and (df["Sector"] == "w").sum() == 0, "ETL slicing bug remains!"
assert df["Police District Name"].isna().sum() == 0 and (df["Police District Name"] == "OTHER").sum() == 0, "Unclean district name!"
assert df["Zip_Code_Clean"].dropna().str.match(r"^\d{5}$").all(), "Non-5-digit Zip_Code_Clean!"
assert df["Place_Group"].isna().sum() == 0, "Missing Place_Group!"
assert (df["Agency_Clean"] == "P").sum() == 0, "Agency_Clean still has 'P'!"
assert df["Block Address"].dropna().str.contains(r"  ", regex=True).sum() == 0, "Double spaces remain in Block Address!"

print("\n[ALL 12 AUTOMATED VERIFICATION ASSERTIONS PASSED SUCCESSFULLY]")

=== EXECUTIVE BEFORE vs AFTER COMPARISON ===
                         Quality Metric  BEFORE (Raw)              AFTER (Cleaned)        Verification Result
      Total Rows (Zero Accidental Loss)       508,106                      508,106             Preserved 100%
                          Total Columns            30 56 (27 raw + 29 clean/flags)   3 dropped, 29 engineered
  Duplicate (Incident ID, Offence Code)             0                            0         Primary key intact
     Null Island Coordinates (0.0, 0.0)        24,648                            0    Converted to NaN + flag
       Negative Durations (Start > End)         8,418                            0        Nullified + flagged
    Negative Dispatch Delays (Main Col)        29,165                            0          Excluded via flag
  Corrupted Beat ('own') / Sector ('w') 1,557 / 1,557                        0 / 0 15 spatial + 1,542 Unknown
     Police District Name NaN / 'OTHER'    1,542 / 15                      


[ALL 12 AUTOMATED VERIFICATION ASSERTIONS PASSED SUCCESSFULLY]


## Step 19 — Export Frozen Cleaned Dataset & Reload Verification

**Why:** Save the cleaned dataset to `data/processed/cleaned_crime_data.csv.gz` (compressed for Git compatibility) and immediately reload a sample to confirm the exported artifact is intact and uncorrupted.

In [19]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False, compression="gzip")

size_mb = OUTPUT_PATH.stat().st_size / 1024 / 1024
print(f"Exported cleaned dataset -> {OUTPUT_PATH.relative_to(REPO_ROOT)} ({size_mb:.2f} MB)")

# Quick reload verification
df_check = pd.read_csv(OUTPUT_PATH, nrows=5, low_memory=False)
assert df_check.shape == (5, df.shape[1]), "Reloaded column count mismatch!"
print(f"Reload check passed: {df_check.shape[1]} columns verified.")
df_check[["Incident ID", "Start_Date_Time", "Police District Number", "State_Clean", "Zip_Code_Clean", "City_Clean", "Place_Group", "Agency_Clean"]]

Exported cleaned dataset -> data/processed/cleaned_crime_data.csv.gz (35.62 MB)
Reload check passed: 56 columns verified.


,Incident ID,Start_Date_Time,Police District Number,State_Clean,Zip_Code_Clean,City_Clean,Place_Group,Agency_Clean
0,201590757,2026-09-14 20:44:00,3D,MD,20904,SILVER SPRING,"Government, Medical & Civic",MCPD
1,201590749,2026-09-14 19:46:00,5D,MD,20874,GERMANTOWN,Retail & Commercial,MCPD
2,201590742,2026-09-14 18:01:00,3D,MD,20904,SILVER SPRING,Residence & Lodging,MCPD
3,201590754,2026-09-14 18:00:00,6D,MD,20879,GAITHERSBURG,Residence & Lodging,MCPD
4,201590766,2026-09-14 17:52:00,3D,MD,20903,SILVER SPRING,Food & Nightlife,MCPD
